# Identify consistent errors and clean them in OY tables

In [60]:
import numpy as np
import os  # For Saving to Folder
import pandas as pd
import matplotlib.pyplot as plt

import socket
import os as os
import sys as sys
import multiprocessing as mp
from pysam import AlignmentFile

### For Arial Font
from matplotlib import rcParams
rcParams['font.family'] = 'sans-serif'   # Set the defaul
### Make sure to have the font installed (it is on cluster for Harald)
rcParams['font.sans-serif'] = ['Arial']

socket_name = socket.gethostname()
print(socket_name)

if socket_name.startswith("compute-"):
    print("HSM Computational partition detected.")
    path = "/n/groups/reich/hringbauer/git/y_chrom/"  # The Path on Midway Cluster
    
elif socket_name.startswith("bionc") or socket_name.startswith("hpc"):
    print("Leipzig Cluster detected!")
    path = "/mnt/archgen/users/hringbauer/git/y_chrom/"
    
else:
    raise RuntimeWarning("Not compatible machine. Check!!")

os.chdir(path)  # Set the right Path (in line with Atom default)

# Show the current working directory. Should be HAPSBURG/Notebooks/ParallelRuns
print(os.getcwd())
print(f"CPU Count: {mp.cpu_count()}")
print(sys.version)

### Custom Imports
from python.pulldown import load_snp_file_ISOGG, call_y_bam, mismatch_path, div_anc_der, get_mismatch_snps, create_parent_dct

hpc030
Leipzig Cluster detected!
/mnt/archgen/users/hringbauer/git/y_chrom
CPU Count: 128
3.12.3 (main, Jan 22 2026, 20:57:42) [GCC 13.3.0]


### 0) Load data for calls

In [72]:
### Load OY SNPs
df1 = pd.read_csv("/mnt/archgen/users/hringbauer/git/y_chrom/data/all_snps_filtered_levels.csv", low_memory=False)
print(f"Loaded {len(df1)} OY SNPs with levels loaded")

### Load OY node dictionary
chpar = create_parent_dct()

### Load "faulty" SNPs to exclude from summary stats
df_cts = pd.read_csv("/mnt/archgen/users/eric_garcia/OYdb/mm12_v3.tsv", sep="\t")
df_ex = df_cts[df_cts["count"]>5] # SNPs that are ancestral in at least 3/12 test sample derived chains

Loaded 2868884 OY SNPs with levels loaded


In [68]:
df1[df1["Y-haplogroup"]=="G-FT49621"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level
2361590,FT50940,Y,18386312,T,C,G-FT49621,NaN,22
2361784,FT51496,Y,22868752,A,G,G-FT49621,NaN,22
2362008,FT48839,Y,3686342,G,T,G-FT49621,NaN,22
2362348,FT50003,Y,9908291,C,G,G-FT49621,NaN,22
2362349,FT50509,Y,16229182,T,C,G-FT49621,NaN,22
2362706,FT49621,Y,7159646,C,A,G-FT49621,NaN,22
2363227,FT50450,Y,15914961,C,T,G-FT49621,NaN,22
2364039,FT48731,Y,3292432,T,C,G-FT49621,NaN,22
2364687,FT49046,Y,4527953,T,A,G-FT49621,NaN,22
2365272,FT51752,Y,28701880,A,C,G-FT49621,NaN,22


# 1) Do manual calls for 12 samples

In [13]:
df_bams = pd.read_csv("/mnt/archgen/users/hringbauer/git/EPIDEMIC/output/tables/autoeager_seqfull_anno.tsv", sep="\t")
print(f"Loaded {len(df_bams)} processed bam files from PTN")

Loaded 414 processed bam files from PTN


In [14]:
### 12 Male individuals in pedigrees
iids = ["PTN209", "PTN211", "PTN139", "PTN146", "PTN267", "PTN238", "PTN126", "PTN406", "PTN231", "PTN495", "PTN390", "PTN439"]
len(iids)

12

In [25]:
for iid in iids:
    print(f"Running IID {iid}...")
    bam_path = df_bams.loc[df_bams["iid"]==iid, "bam"].values[0]

    ### Do the call
    df_ch, df_der = call_y_bam(df=df1, path_bam=bam_path,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

    ### Post-process output for info for manual call
    dft = div_anc_der(df_ch)
    dfd =dft[dft["Derived"]>dft["Ancestral"]]
    display(dfd.sort_values(by="#DER in par.").tail(20))      

Running IID PTN209...
Average Coverage: 7.2976x
#Sites covered: 2370738/2868884
#Derived Loci: 
6390 / 2370738 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
84668,R-M269,27,97,0,97,0,29,589
1448,C-V9,7,38,3,35,0,286,656
84564,R-L23,28,3,0,3,0,29,686
84587,R-L51,29,5,0,5,0,29,689
55893,R-BY3293,35,1,0,1,0,43,689
84552,R-L151,31,3,0,3,0,29,694
84890,R-PF6538,31,1,0,1,0,29,694
50798,R-BY1188,53,1,0,1,0,52,697
84558,R-L2,36,1,0,1,0,29,697
56679,R-BY3953,45,1,0,1,0,39,697


Running IID PTN211...
Average Coverage: 0.3651x
#Sites covered: 820913/2868884
#Derived Loci: 
4017 / 820913 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
55679,R-Y151368,45,1,0,1,0,24,249
41461,R-FT127736,45,1,0,1,0,18,249
56196,R-Y22894,44,1,0,1,0,7,249
45158,R-FT371228,48,1,0,1,0,9,249
54600,R-MF7792,46,1,0,1,0,9,249
38009,R-BY6971,47,1,0,1,0,20,249
35613,R-BY33664,47,1,0,1,0,11,249
58766,R-Z42134,42,1,0,1,0,12,249
58593,R-Z23516,42,1,0,1,0,13,249
42978,R-FT211469,41,1,0,1,0,11,249


Running IID PTN139...
Average Coverage: 0.3297x
#Sites covered: 752016/2868884
#Derived Loci: 
3882 / 752016 covered>0


...

,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
41345,R-FT34517,44,1,0,1,0,14,215
30897,R-BY166488,48,1,0,1,0,8,215
41804,R-FT375991,47,1,0,1,0,10,215
46080,R-FTB93188,49,1,0,1,0,8,215
39843,R-FT21699,50,1,0,1,0,14,215
40555,R-FT263049,51,1,0,1,0,13,215
40323,R-FT246039,51,1,0,1,0,10,215
33010,R-BY35726,51,1,0,1,0,10,215
39238,R-FT185374,55,1,0,1,0,10,215
40873,R-FT28897,44,1,0,1,0,9,215


Running IID PTN390...
Average Coverage: 0.3510x
#Sites covered: 784974/2868884
#Derived Loci: 
4527 / 784974 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
4738,E-P2,12,2,0,2,0,5,88
4635,E-M35,14,32,0,32,0,5,90
210,A-Y8284,5,3,0,3,0,36,99
466,BT-M91,6,137,16,119,2,36,102
3904,E-FTB9206,26,1,0,1,0,63,122
5288,E-Y228679,29,1,0,1,0,32,122
4667,E-M78,16,20,0,20,0,5,122
5239,E-Y216284,24,1,0,1,0,35,122
5736,E-Z1919,18,1,0,1,0,5,142
2224,E-BY8085,33,1,0,1,0,36,143


Running IID PTN439...
Average Coverage: 0.3532x
#Sites covered: 799009/2868884
#Derived Loci: 
3146 / 799009 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
34982,R-BY3508,38,1,0,1,0,128,104
38849,R-FGC12045,40,1,0,1,0,133,104
41197,R-FT157508,53,1,0,1,0,152,104
4745,E-P177,11,2,0,2,0,6,107
4746,E-P2,12,5,0,5,0,6,109
4750,E-PF1885,12,1,0,1,0,6,109
221,A-Y8284,5,4,0,4,0,54,113
4643,E-M35,14,36,0,36,0,6,114
486,BT-M91,6,127,15,111,1,54,117
4677,E-M78,16,15,0,15,0,6,150


# Hard-code manual calls
Read-out from the above output

In [26]:
y_dct = {
    "PTN209": "R-FTG53091",
    "PTN211": "R-FTG53091",
    "PTN139": "I-FT58623",
    "PTN146": "I-FT58623",
    "PTN267": "I-FT58623",
    "PTN238": "I-DF29",
    "PTN126": "I-FT9196",
    "PTN406": "I-S6277",
    "PTN231": "R-BY61482",
    "PTN495": "R-BY61482",
    "PTN390": "E-S2979",
    "PTN439": "E-S2979"
}

### Extract Mismatching SNPs

In [35]:
%%time

df_mms_lst  = []

for iid, hpgrp in y_dct.items():
    print(f"Running IID {iid}...")
    bam_path = df_bams.loc[df_bams["iid"]==iid, "bam"].values[0]

    ### Get allele counts
    df_ch, df_der = call_y_bam(df=df1, path_bam=bam_path,
                           path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 
    ### Get mismatches
    df_mms = get_mismatch_snps(hpgrp, chpar=chpar, df_ch=df_ch)

    df_mms_lst.append(df_mms)

Running IID PTN209...
Average Coverage: 7.2976x
#Sites covered: 2370738/2868884
#Derived Loci: 
6390 / 2370738 covered>0
Running IID PTN211...
Average Coverage: 0.3651x
#Sites covered: 820913/2868884
#Derived Loci: 
4017 / 820913 covered>0
Running IID PTN139...
Average Coverage: 0.3297x
#Sites covered: 752016/2868884
#Derived Loci: 
3882 / 752016 covered>0
Running IID PTN146...
Average Coverage: 0.3544x
#Sites covered: 792820/2868884
#Derived Loci: 
4338 / 792820 covered>0
Running IID PTN267...
Average Coverage: 0.4835x
#Sites covered: 1004319/2868884
#Derived Loci: 
4110 / 1004319 covered>0
Running IID PTN238...
Average Coverage: 0.2982x
#Sites covered: 694724/2868884
#Derived Loci: 
3436 / 694724 covered>0
Running IID PTN126...
Average Coverage: 0.3789x
#Sites covered: 835624/2868884
#Derived Loci: 
5023 / 835624 covered>0
Running IID PTN406...
Average Coverage: 0.3987x
#Sites covered: 878674/2868884
#Derived Loci: 
4372 / 878674 covered>0
Running IID PTN231...
Average Coverage: 0.35

### Analyze Mismatching SNPs

In [41]:
mms = list(map(len,df_mms_lst))
mms

[29, 6, 11, 7, 16, 9, 10, 7, 14, 9, 5, 6]

In [48]:
print(f"{np.mean(mms[1:]):.3f}") ### Average Mismatches in 1x Data

9.091


In [49]:
df_mms = pd.concat(df_mms_lst)

In [70]:
df_cts = pd.value_counts(df_mms["Subgroup Name"])

/tmp/ipykernel_1938581/1535705838.py:1: FutureWarning: pandas.value_counts is deprecated and will be removed in a future version. Use pd.Series(obj).value_counts() instead.
  df_cts = pd.value_counts(df_mms["Subgroup Name"])


In [71]:
df_cts.to_csv("/mnt/archgen/users/hringbauer/git/y_chrom/data/OY/mm12.tsv", sep="\t")

In [62]:
df_mms

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
2127395,"MF784814,YSC0000166",Y,14116584,A,T,R-P297,NaN,26,17,0,0,0,17,0
2090274,"FGC58,MF803532",Y,7100362,T,C,R-M343,NaN,22,0,0,0,8,8,0
2090443,"FGC66,MF803527",Y,7081561,T,C,R-M343,NaN,22,0,0,0,9,9,0
2073713,"FGC280,MF786133",Y,19298321,A,G,R-UTY2,NaN,20,9,0,0,0,9,0
2075838,"MF789201,YSC0000067",Y,7133986,C,G,R-UTY2,NaN,20,0,8,0,0,8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
684450,"Y9964,Z9440",Y,18132698,T,C,A-V168,NaN,3,0,0,0,1,1,0
684580,CTS12370,Y,28583229,T,C,A-V168,NaN,3,0,0,0,1,1,0
684129,BY184462,Y,8055446,G,A,A-L1090,NaN,2,0,0,1,0,1,0
684138,FGC27824,Y,23139472,T,C,A-L1090,NaN,2,0,0,0,2,2,0


# 3) Try new single samples

In [23]:
df_bams = pd.read_csv("/mnt/archgen/users/hringbauer/git/EPIDEMIC/output/tables/autoeager_seqfull_anno.tsv", sep="\t")
print(f"Loaded {len(df_bams)} processed bam files from PTN")
df_m = df_bams[df_bams["sex"]=="m"]
print(f"Extracted {len(df_m)} males")

Loaded 414 processed bam files from PTN
Extracted 209 males


In [ ]:
df_m["iid"][-50:] # Plot a few male IIDs

### 3) Analyze single example

In [73]:
iid = "PTN238"
print(f"Running IID {iid}...")
bam_path = df_bams.loc[df_bams["iid"]==iid, "bam"].values[0]

### Do the call
df_ch, df_der = call_y_bam(df=df1, path_bam=bam_path,
                       path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

### Post-process output for info for manual call
dft = div_anc_der(df_ch, df_exclude=df_ex)
dfd = dft[dft["Derived"]>dft["Ancestral"]]
display(dfd.sort_values(by="#DER in par.").tail(30))  

Running IID PTN238...
Average Coverage: 0.2982x
#Sites covered: 694724/2868884
#Derived Loci: 
3436 / 694724 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
43423,R-FTA27308,42,1,0,1,0,72,124
24126,N-BY99399,53,1,0,1,0,31,124
42590,R-FT67224,55,1,0,1,0,78,124
43999,R-FTA70641,50,1,0,1,0,80,124
43554,R-FTA32861,52,1,0,1,0,78,124
45368,R-FTB75363,54,1,0,1,0,87,124
28362,R-A12274,53,1,0,1,0,84,124
25774,O-CTS11725,27,1,0,1,0,66,125
14043,I-M170,14,33,0,33,0,0,127
18547,J-FT36487,46,1,0,1,0,33,127


In [57]:
dfmm = get_mismatch_snps("I-M253", chpar=chpar, df_ch=df_ch)

In [58]:
dfmm

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
600278,PF3518,Y,6607318,C,T,IJ-P124,NaN,12,0,1,0,0,1,0
597522,CTS5750,Y,16467111,T,C,F-M89,F,8,0,0,0,1,1,0
597760,"MF808158,PF1911",Y,23729951,T,C,F-M89,F,8,0,0,0,1,1,0
597769,"MF806436,PF1720,TY199699",Y,17142068,T,A,F-M89,F,8,0,0,0,1,1,0
596861,Y1445,Y,2987520,C,T,CT-M168,NaN,6,0,2,0,1,2,1
595966,"V6478,Z9327",Y,17028360,T,A,A-V168,NaN,3,0,0,0,2,2,0
596059,"Y9964,Z9440",Y,18132698,T,C,A-V168,NaN,3,0,0,0,1,1,0
595796,FGC27794,Y,23049484,T,C,A-L1090,NaN,2,0,0,0,2,2,0
595851,V3167,Y,15833573,T,C,A-L1090,NaN,2,0,0,0,1,1,0


In [45]:
dfmm[~dfmm["Subgroup Name"].isin(df_ex["Subgroup Name"])]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
564489,"L777,YSC0000248",Y,13657777,T,C,R-M269,NaN,27,0,0,0,1,1,0
550067,"FGC66,MF803527",Y,7081561,T,C,R-M343,NaN,22,0,0,0,1,1,0
533546,A5111,Y,13511050,C,G,A-L1090,NaN,2,0,1,0,0,1,0


In [59]:
df_ch[df_ch["Y-haplogroup"]=="I-M253"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#
604184,"Page123,PAGES00123,Z2763",Y,14898383,G,A,I-M253,NaN,17,1,0,0,0,0,1
605549,Z2860,Y,22480517,C,A,I-M253,NaN,17,1,0,0,0,0,1


##

### Kamenice IID KNC002

In [68]:
### Do the call
df_ch, df_der = call_y_bam(df=df1, path_bam="/mnt/archgen/Autorun_eager/eager_outputs/RM/KNC/KNC139/trimmed_bam/KNC139_ss.A0101_udgnone.trimmed.bam",
                       path_bed='/mnt/archgen/users/hringbauer/git/y_chrom/data/OY_snps.bed') 

### Post-process output for info for manual call
dft = div_anc_der(df_ch, df_exclude=df_ex)
dfd =dft[dft["Derived"]>dft["Ancestral"]]
display(dfd.sort_values(by="#DER in par.").tail(30))  

Average Coverage: 1.2350x
#Sites covered: 889590/2868884
#Derived Loci: 
8264 / 889590 covered>0


,Branch,Level,Total_SNPs,Ancestral,Derived,Uncovered,#ANC in par.,#DER in par.
52781,R-FTC9482,56,1,0,1,0,24,352
53490,R-FTD61432,56,1,0,1,0,11,352
59648,R-Y81029,62,2,0,1,1,22,352
58596,R-Y331948,56,1,0,1,0,23,352
32481,R-A6097,50,1,0,1,0,15,352
56895,R-S3253,50,1,0,1,0,14,352
41211,R-DC858,51,1,0,1,0,13,352
34229,R-BY149318,51,1,0,1,0,15,352
45529,R-FT25395,51,1,0,1,0,5,352
48842,R-FT99153,51,1,0,1,0,21,352


In [ ]:
R-Y305501 III KNC002, KNC006, KNC010, KNC139
R-Y439571 

In [ ]:
dfmm = get_mismatch_snps("R-Y583795", chpar=chpar, df_ch=df_ch)

In [66]:
df_ch[df_ch["Y-haplogroup"]=="R-Y305501"]

,Subgroup Name,chrom,pos,ref,alt,Y-haplogroup,YFull translation,Level,A,C,G,T,ref#,alt#


# Area51

In [20]:
df_m[df_m["iid"].str.contains("BMG001")]["bam"].values[0]

'/mnt/archgen/Autorun_eager/eager_outputs/SG/BMG/BMG001/merged_bams/additional/BMG001_ss_libmerged_add.bam'

In [ ]:
WSQ003

In [79]:
def exclude_snps(df_ch, df_ex, verbose=False, col="Subgroup Name"):
    """Filter SNPs from df_ch in df_ex"""

    idx= df_ch[col].isin(df_ex[col])
    df_ch2 = df_ch[~idx].copy()
    if verbose:
        print(f"Filtered to {len(df_ch2)}/{len(df_ch)}")
    return df_ch2

In [81]:
df_ch2 = exclude_snps(df_ch, df_ex, verbose=True)

Filtered to 859174/859183


In [ ]:
df_b

In [85]:
dfs[dfs["snp"].str.contains("rs5743618")]

,snp


In [86]:
dfs

,snp
0,rs12562034
1,rs3934834
2,rs9442372
3,rs3737728
4,rs6687776
...,...
475801,rs5940484
475802,rs5983658
475803,rs553678
475804,rs473491
